#Set up


**Authorise colab to connect to google drive**

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

Mounted at /content/drive


**Libraries**

In [2]:
pip install linearmodels

   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.5 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.5 MB ? eta -:--:--
   -------------------- ------------------- 0.8/1.5 MB 931.2 kB/s eta 0:00:01
   --------------------------- ------------ 1.0/1.5 MB 1.1 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 1.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.5 MB 2.8 MB/s eta 0:00:04
   ---- ----------------------------------- 1.0/9.5 MB 2.4 MB/s eta 0:00:04
   ------- -------------------------------- 1.8/9.5 MB 2.9 MB/s eta 0:00:03
   ---------- ----------------------------- 2.6/9.5 MB 3.3 MB/s eta 0:00:03
   --------------- ------------------------ 3.7/9.5 MB 3.6 MB/s eta 0:00:02
   ------------------ --------------------- 4


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import numpy as np
import pandas as pd

#for checking results
import statsmodels.tsa.api as tsa
import statsmodels.api as sm
from linearmodels.panel import PanelOLS
from linearmodels.panel import RandomEffects

#Data Import

construct a file called Ectr3_data with the assignment data in it in your drive for the path to work

In [5]:
file_path = 'C:/Users/Sara/Downloads/Econometrics III/data_assignment1.csv'
df = pd.read_csv(file_path)

In [6]:
#data structure
df.head()

,Unnamed: 0,X,exp,wks,bluecol,ind,south,smsa,married,gender,union,edu,col,lwage
0,1,1,3,32,0,0,1,0,1,0,0,9,0,5.56068
1,2,2,4,43,0,0,1,0,1,0,0,9,0,5.72031
2,3,3,5,40,0,0,1,0,1,0,0,9,0,5.99645
3,4,4,6,39,0,0,1,0,1,0,0,9,0,5.99645
4,5,5,7,42,0,1,1,0,1,0,0,9,0,6.06146


In [ ]:
#drop redundant columns
df.drop(columns=['Unnamed: 0', 'X'], inplace=True)

#construct individual and year indicators
N = 595
T = 7

year_range = [i for i in range(1976, 1976 + 7)]
df['year'] = year_range * N

i_indicator = []
for i in range(1, 1 + 595):
  indv_i = [i]
  i_indicator += indv_i * T
df['i'] = i_indicator

df.head(14)

,exp,wks,bluecol,ind,south,smsa,married,gender,union,edu,col,lwage,year,i
0,3,32,0,0,1,0,1,0,0,9,0,5.56068,1976,1
1,4,43,0,0,1,0,1,0,0,9,0,5.72031,1977,1
2,5,40,0,0,1,0,1,0,0,9,0,5.99645,1978,1
3,6,39,0,0,1,0,1,0,0,9,0,5.99645,1979,1
4,7,42,0,1,1,0,1,0,0,9,0,6.06146,1980,1
5,8,35,0,1,1,0,1,0,0,9,0,6.17379,1981,1
6,9,32,0,1,1,0,1,0,0,9,0,6.24417,1982,1
7,30,34,1,0,0,0,1,0,0,11,0,6.16331,1976,2
8,31,27,1,0,0,0,1,0,0,11,0,6.21461,1977,2
9,32,33,1,1,0,0,1,0,1,11,0,6.26340,1978,2


In [ ]:
df.describe()

,exp,wks,bluecol,ind,south,smsa,married,gender,union,edu,col,lwage,year,i
count,4165.000000,4165.000000,4165.000000,4165.000000,4165.000000,4165.000000,4165.000000,4165.000000,4165.000000,4165.000000,4165.000000,4165.000000,4165.00000,4165.000000
mean,19.853782,46.811525,0.511164,0.395438,0.290276,0.653782,0.814406,0.112605,0.363986,12.845378,0.072269,6.676346,1979.00000,298.000000
std,10.966370,5.129098,0.499935,0.489003,0.453944,0.475821,0.388826,0.316147,0.481202,2.787995,0.258964,0.461512,2.00024,171.782086
min,1.000000,5.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,4.000000,0.000000,4.605170,1976.00000,1.000000
25%,11.000000,46.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,12.000000,0.000000,6.395260,1977.00000,149.000000
50%,18.000000,48.000000,1.000000,0.000000,0.000000,1.000000,1.000000,0.000000,0.000000,12.000000,0.000000,6.684610,1979.00000,298.000000
75%,29.000000,50.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,1.000000,16.000000,0.000000,6.952730,1981.00000,447.000000
max,51.000000,52.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,17.000000,1.000000,8.537000,1982.00000,595.000000


#Pre-Assignment (1) - Regression for each year

## Manual

Estimate beta

In [ ]:
beta_hats = {}
beta_se = {}
beta_t = {}
for year in range(1976, 1976 + 7):
  df_year = df[df['year'] == year].drop(columns=['year', 'i'])
  y = df_year['lwage'].to_numpy()
  X = df_year.drop(columns=['lwage']).to_numpy()
  X = np.c_[np.ones(len(X)), X] #add constant
  beta_hats_year = np.linalg.inv(X.T @ X) @ X.T @ y
  u = y - X @ beta_hats_year
  N = X.shape[0]
  k = X.shape[1]
  var_u = (1/(N-k)) * (u.T @ u)
  beta_var_mat = var_u * np.linalg.inv(X.T @ X)
  beta_se_year = np.sqrt(np.diag(beta_var_mat))
  beta_t_year = beta_hats_year / beta_se_year

  beta_hats[f'beta_hats_{year}'] = beta_hats_year
  beta_se[f'beta_se_{year}'] = beta_se_year
  beta_t[f'beta_t_{year}'] = beta_t_year

In [ ]:
table_1_dict = {'variables': ['const'] + df.drop(columns = ['year', 'i', 'lwage']).columns.tolist()}

for year in range(1976, 1976+7):
    table_1_dict[f'{year} beta'] = beta_hats[f'beta_hats_{year}']
    table_1_dict[f'{year} se'] = beta_se[f'beta_se_{year}']
    table_1_dict[f'{year} t'] = beta_t[f'beta_t_{year}']

table_1 = pd.DataFrame(table_1_dict)
table_1_rounded = table_1.round(3)
table_1_rounded

,variables,1976 beta,1976 se,1976 t,1977 beta,1977 se,1977 t,1978 beta,1978 se,1978 t,...,1979 t,1980 beta,1980 se,1980 t,1981 beta,1981 se,1981 t,1982 beta,1982 se,1982 t
0,const,5.203,0.134,38.871,5.611,0.138,40.651,5.707,0.189,30.175,...,28.037,5.561,0.171,32.488,5.572,0.176,31.635,5.850,0.180,32.546
1,exp,0.010,0.001,8.698,0.008,0.001,7.102,0.008,0.001,5.743,...,5.081,0.006,0.001,4.894,0.006,0.001,4.462,0.005,0.001,3.661
2,wks,0.006,0.002,3.128,0.001,0.002,0.313,0.000,0.003,0.072,...,3.285,0.006,0.003,2.228,0.005,0.003,1.887,0.003,0.003,1.066
3,bluecol,-0.126,0.031,-4.140,-0.105,0.029,-3.668,-0.162,0.038,-4.283,...,-3.868,-0.167,0.035,-4.763,-0.131,0.036,-3.664,-0.165,0.037,-4.415
4,ind,0.020,0.025,0.797,0.018,0.023,0.755,0.044,0.031,1.445,...,2.147,0.087,0.027,3.173,0.091,0.028,3.194,0.090,0.029,3.065
5,south,-0.056,0.026,-2.108,-0.059,0.025,-2.378,-0.053,0.033,-1.627,...,-2.069,-0.054,0.029,-1.850,-0.055,0.030,-1.817,-0.058,0.031,-1.845
6,smsa,0.181,0.026,6.990,0.156,0.024,6.405,0.150,0.031,4.832,...,4.764,0.163,0.028,5.722,0.174,0.029,6.042,0.162,0.030,5.427
7,married,0.094,0.046,2.059,0.071,0.042,1.694,0.069,0.052,1.326,...,1.802,0.096,0.048,1.992,0.151,0.051,2.961,0.104,0.049,2.112
8,gender,-0.290,0.055,-5.283,-0.346,0.051,-6.827,-0.425,0.064,-6.637,...,-6.158,-0.365,0.058,-6.284,-0.264,0.062,-4.258,-0.318,0.061,-5.181
9,union,0.121,0.027,4.492,0.109,0.026,4.245,0.063,0.033,1.907,...,1.893,0.080,0.030,2.694,0.097,0.031,3.121,0.112,0.032,3.498


## With packages for checking accuracy of results

In [ ]:
OLS_by_year = {}
for year in range(1976, 1976 + 7):
  df_year = df[df['year'] == year]
  X = df_year.drop(columns=['year', 'i', 'lwage'])
  X = sm.add_constant(X)
  y = df_year['lwage']
  res = sm.OLS(y, X).fit() #non-robust specification
  OLS_by_year[f'ols_{year}'] = res

In [ ]:
for year in range(1976, 1976 + 7):
  print(f'Year {year}')
  print(OLS_by_year[f'ols_{year}'].summary())

Year 1976
                            OLS Regression Results                            
Dep. Variable:                  lwage   R-squared:                       0.489
Model:                            OLS   Adj. R-squared:                  0.479
Method:                 Least Squares   F-statistic:                     50.71
Date:                Wed, 18 Feb 2026   Prob (F-statistic):           1.03e-77
Time:                        09:01:44   Log-Likelihood:                -81.393
No. Observations:                 595   AIC:                             186.8
Df Residuals:                     583   BIC:                             239.4
Df Model:                          11                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          5.2034      0.134     38.87

In [ ]:
table_1_rounded

,variables,1976 beta,1976 se,1976 t,1977 beta,1977 se,1977 t,1978 beta,1978 se,1978 t,...,1979 t,1980 beta,1980 se,1980 t,1981 beta,1981 se,1981 t,1982 beta,1982 se,1982 t
0,const,5.203,0.134,38.871,5.611,0.138,40.651,5.707,0.189,30.175,...,28.037,5.561,0.171,32.488,5.572,0.176,31.635,5.850,0.180,32.546
1,exp,0.010,0.001,8.698,0.008,0.001,7.102,0.008,0.001,5.743,...,5.081,0.006,0.001,4.894,0.006,0.001,4.462,0.005,0.001,3.661
2,wks,0.006,0.002,3.128,0.001,0.002,0.313,0.000,0.003,0.072,...,3.285,0.006,0.003,2.228,0.005,0.003,1.887,0.003,0.003,1.066
3,bluecol,-0.126,0.031,-4.140,-0.105,0.029,-3.668,-0.162,0.038,-4.283,...,-3.868,-0.167,0.035,-4.763,-0.131,0.036,-3.664,-0.165,0.037,-4.415
4,ind,0.020,0.025,0.797,0.018,0.023,0.755,0.044,0.031,1.445,...,2.147,0.087,0.027,3.173,0.091,0.028,3.194,0.090,0.029,3.065
5,south,-0.056,0.026,-2.108,-0.059,0.025,-2.378,-0.053,0.033,-1.627,...,-2.069,-0.054,0.029,-1.850,-0.055,0.030,-1.817,-0.058,0.031,-1.845
6,smsa,0.181,0.026,6.990,0.156,0.024,6.405,0.150,0.031,4.832,...,4.764,0.163,0.028,5.722,0.174,0.029,6.042,0.162,0.030,5.427
7,married,0.094,0.046,2.059,0.071,0.042,1.694,0.069,0.052,1.326,...,1.802,0.096,0.048,1.992,0.151,0.051,2.961,0.104,0.049,2.112
8,gender,-0.290,0.055,-5.283,-0.346,0.051,-6.827,-0.425,0.064,-6.637,...,-6.158,-0.365,0.058,-6.284,-0.264,0.062,-4.258,-0.318,0.061,-5.181
9,union,0.121,0.027,4.492,0.109,0.026,4.245,0.063,0.033,1.907,...,1.893,0.080,0.030,2.694,0.097,0.031,3.121,0.112,0.032,3.498


#Pre-Assignment (2) - Pooled

##Manual

In [ ]:
df_pooled = df.drop(columns=['year', 'i'])

In [ ]:
y = df_pooled['lwage'].to_numpy()
X = df_pooled.drop(columns=['lwage']).to_numpy()
X = np.c_[np.ones(len(X)), X]
beta_hats_pooled = np.linalg.inv(X.T @ X) @ X.T @ y

u = y - X @ beta_hats_pooled
N = X.shape[0]
k = X.shape[1]
var_u = (1/(N-k)) * (u.T @ u)
beta_var_mat = var_u * np.linalg.inv(X.T @ X)
beta_se_pooled = np.sqrt(np.diag(beta_var_mat))

beta_t_pooled = beta_hats_pooled / beta_se_pooled

In [ ]:
table_2 = pd.DataFrame({'variables': ['const'] + df_pooled.drop(columns = ['lwage']).columns.tolist(),
                        'beta_hats': beta_hats_pooled,
                        'se': beta_se_pooled,
                        't': beta_t_pooled})
table_2_rounded = table_2.round(3)
table_2_rounded

,variables,beta_hats,se,t
0,const,5.441,0.072,75.901
1,exp,0.010,0.001,19.372
2,wks,0.005,0.001,4.471
3,bluecol,-0.149,0.015,-9.913
4,ind,0.053,0.012,4.397
5,south,-0.053,0.013,-4.149
6,smsa,0.145,0.012,11.767
7,married,0.066,0.021,3.144
8,gender,-0.353,0.026,-13.761
9,union,0.102,0.013,7.800


##with Packages

In [ ]:
y = df_pooled['lwage']
X = df_pooled.drop(columns=['lwage'])
X = sm.add_constant(X)
ols_pooled = sm.OLS(y, X).fit()
print(ols_pooled.summary())

                            OLS Regression Results                            
Dep. Variable:                  lwage   R-squared:                       0.401
Model:                            OLS   Adj. R-squared:                  0.399
Method:                 Least Squares   F-statistic:                     252.6
Date:                Wed, 18 Feb 2026   Prob (F-statistic):               0.00
Time:                        09:01:45   Log-Likelihood:                -1621.9
No. Observations:                4165   AIC:                             3268.
Df Residuals:                    4153   BIC:                             3344.
Df Model:                          11                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          5.4412      0.072     75.901      0.0

#(a) Between Estimation

##Manual

In [ ]:
N = 595
T = 7
D = np.kron(np.identity(N), np.ones((T,1)))
P_D = D @ np.linalg.inv(D.T @ D) @ D.T

Transform data to averages

In [ ]:
y = df['lwage'].to_numpy()
X = df.drop(columns=['lwage', 'year', 'i']).to_numpy()
X = np.c_[np.ones(len(X)), X] #add constant
y_avg = P_D @ y
X_avg = P_D @ X

between estimation betas

In [ ]:
beta_hats_btwn = np.linalg.inv(X_avg.T @ X_avg) @ X_avg.T @ y_avg

In [ ]:
alpha_i = y_avg - X_avg @ beta_hats_btwn

In [ ]:
k = X.shape[1]
var_alpha_i = (1/(N-k)) * sum(alpha_i**2)

In [ ]:
#First unknown variance: variance of alpha
var_alpha_i

np.float64(0.5304178375964181)

SE of between estimation betas

In [ ]:
var_matrix_beta_btwn = var_alpha_i * np.linalg.inv(X_avg.T @ X_avg)
beta_se_btwn = np.sqrt(np.diag(var_matrix_beta_btwn))
beta_t_btwn = beta_hats_btwn / beta_se_btwn

compile results

In [ ]:
print("estimated variance of alpha_i using between estimation:")
print(f'{var_alpha_i}')

table_2 = pd.DataFrame({
    'variables': ['const'] + df.drop(columns = ['year', 'i', 'lwage']).columns.tolist(),
    'beta_hats': beta_hats_btwn,
    'se': beta_se_btwn,
    't': beta_t_btwn
})
table_2_rounded = table_2.round(3)
table_2_rounded


estimated variance of alpha_i using between estimation:
0.5304178375964181


,variables,beta_hats,se,t
0,const,5.263,0.207,25.381
1,exp,0.007,0.001,6.084
2,wks,0.010,0.004,2.751
3,bluecol,-0.176,0.035,-5.081
4,ind,0.064,0.026,2.433
5,south,-0.055,0.027,-2.068
6,smsa,0.170,0.026,6.470
7,married,0.135,0.049,2.769
8,gender,-0.300,0.056,-5.364
9,union,0.119,0.030,3.967


##with packages

In [ ]:
df_avg = df.groupby('i').mean()
y_bar = df_avg['lwage']
X_bar = df_avg.drop(columns=['year', 'lwage'])
X_bar = sm.add_constant(X_bar)
ols_btwn = sm.OLS(y_bar, X_bar).fit()
print(ols_btwn.summary())

                            OLS Regression Results                            
Dep. Variable:                  lwage   R-squared:                       0.521
Model:                            OLS   Adj. R-squared:                  0.512
Method:                 Least Squares   F-statistic:                     57.76
Date:                Wed, 18 Feb 2026   Prob (F-statistic):           6.46e-86
Time:                        09:01:47   Log-Likelihood:                -70.657
No. Observations:                 595   AIC:                             165.3
Df Residuals:                     583   BIC:                             218.0
Df Model:                          11                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          5.2634      0.207     25.381      0.0

#(b) Within Estimation

##Manual

since we will be performing demeaning, time invariant variables will fall out, we first take a look at what these variables are

In [ ]:
#is a certain variable constant across time for all individuals?
time_inv_varb = []
for varb in df.drop(columns = ['lwage', 'year', 'i']).columns:
  t_invariant_each_i = []
  for i in range(1, 1 + 595):
    varb_i = df[df['i']==i][varb]
    if len(varb_i.unique()) == 1:
      t_invariant_each_i.append(True)
    else:
      t_invariant_each_i.append(False)

  if len(t_invariant_each_i) != 595:
    print('error')

  if all(t_invariant_each_i):
    time_inv_varb.append(varb)
    print(f'{varb} is time invariant across individuals')
  else:
    print(f'{varb} is NOT time invariant across individuals')


print(f'time invariant variables: {time_inv_varb}')

exp is NOT time invariant across individuals
wks is NOT time invariant across individuals
bluecol is NOT time invariant across individuals
ind is NOT time invariant across individuals
south is NOT time invariant across individuals
smsa is NOT time invariant across individuals
married is NOT time invariant across individuals
gender is time invariant across individuals
union is NOT time invariant across individuals
edu is time invariant across individuals
col is time invariant across individuals
time invariant variables: ['gender', 'edu', 'col']


we need to remove the columns for constant, gender, edu, and col

In [ ]:
N = 595
T = 7
M_D = np.identity(N*T) - P_D
#redefine X and we don't add constant
X = df.drop(columns=['year', 'i', 'lwage'] + time_inv_varb)
X_column_order = X.columns #record order of columns (time variant variables)
X = X.to_numpy()
beta_hats_wthn = np.linalg.inv(X.T @ M_D @ X) @ X.T @ M_D @ y

In [ ]:
wthn_resids = (M_D @ y) - (M_D @ X) @ beta_hats_wthn
k = X.shape[1]
var_eta = (1/(N * (T-1) - k)) * sum(wthn_resids ** 2)

In [ ]:
var_matrix_beta_wthn = var_eta * np.linalg.inv(X.T @ M_D @ X)
beta_se_wthn = np.sqrt(np.diag(var_matrix_beta_wthn))
beta_t_wthn = beta_hats_wthn / beta_se_wthn

Results

In [ ]:
print("estimated variance of eta_i using within estimation:")
print(f'{var_eta}')
table_3 = pd.DataFrame({
    'variables': X_column_order,
    'beta_hats': beta_hats_wthn,
    'beta_se': beta_se_wthn,
    'beta_t': beta_t_wthn
})
table_3.round(3)


estimated variance of eta_i using within estimation:
0.023476664933713053


,variables,beta_hats,beta_se,beta_t
0,exp,0.097,0.001,81.099
1,wks,0.001,0.001,1.894
2,bluecol,-0.025,0.014,-1.790
3,ind,0.021,0.016,1.333
4,south,-0.003,0.035,-0.092
5,smsa,-0.044,0.020,-2.233
6,married,-0.030,0.019,-1.581
7,union,0.034,0.015,2.271


##with packages

In [ ]:
df_panel = df
df_panel = df_panel.set_index(['i', 'year'])
y_wthn = df_panel['lwage']
X_wthn = df_panel.drop(columns=['lwage'])
res_wthn = PanelOLS(y_wthn, X_wthn, entity_effects=True, drop_absorbed=True).fit()
print(res_wthn.summary)

/tmp/ipython-input-122724339.py:5: AbsorbingEffectWarning: 
Variables have been fully absorbed and have removed from the regression:

gender, edu, col

  res_wthn = PanelOLS(y_wthn, X_wthn, entity_effects=True, drop_absorbed=True).fit()


                          PanelOLS Estimation Summary                           
Dep. Variable:                  lwage   R-squared:                        0.6525
Estimator:                   PanelOLS   R-squared (Between):              0.4704
No. Observations:                4165   R-squared (Within):               0.6525
Date:                Wed, Feb 18 2026   R-squared (Overall):              0.4706
Time:                        09:01:56   Log-likelihood                    2228.8
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      836.08
Entities:                         595   P-value                           0.0000
Avg Obs:                       7.0000   Distribution:                  F(8,3562)
Min Obs:                       7.0000                                           
Max Obs:                       7.0000   F-statistic (robust):             836.08
                            

#(c) FGLS

## Manual - attempt one

In [ ]:
#var_alpha_i adjustment due to small T
var_alpha_i_adj = (1/N) * sum(alpha_i**2) - (1/T) * var_eta

theta_hat = np.sqrt(var_eta / (T*var_alpha_i_adj + var_eta))
X_i_asterisk = {}
y_i_asterisk = {}
for i in range(1, 1+595):
  df_i = df[df['i'] == i]
  X_i = df_i.drop(columns=['year', 'i', 'lwage']).to_numpy()
  X_i = np.c_[np.ones(len(X_i)), X_i]
  y_i = df_i['lwage'].to_numpy()
  X_i_bar = ((1/T) * np.ones((T,T))) @ X_i
  y_i_bar = ((1/T) * np.ones((T,T))) @ y_i
  X_i_asterisk[f'X_{i}_asterisk'] = (1/np.sqrt(var_eta)) * (X_i - (1-theta_hat) * X_i_bar)
  y_i_asterisk[f'y_{i}_asterisk'] = (1/np.sqrt(var_eta)) * (y_i - (1-theta_hat) * y_i_bar)

In [ ]:
k = X_i.shape[1]
temp_1 = np.zeros((k,k))
for i in range(1, 1+595):
  temp_1 += X_i_asterisk[f'X_{i}_asterisk'].T @ X_i_asterisk[f'X_{i}_asterisk']
temp_2 = np.zeros(k)
for i in range(1, 1+595):
  temp_2 += (X_i_asterisk[f'X_{i}_asterisk'].T @ y_i_asterisk[f'y_{i}_asterisk'])
beta_hats_fgls = np.linalg.inv(temp_1) @ temp_2
var_matrix_beta_fgls = np.linalg.inv(temp_1)
beta_se_fgls = np.sqrt(np.diag(var_matrix_beta_fgls))
beta_t_fgls = beta_hats_fgls / beta_se_fgls

In [ ]:
print("adjust variance of alpha_i using variance of eta_i")
print(f'{var_alpha_i_adj}')
table_4 = pd.DataFrame({
    'variables': ['const'] + df_i.drop(columns = ['year', 'i', 'lwage']).columns.tolist(),
    'beta_hats': beta_hats_fgls,
    'beta_se': beta_se_fgls,
    'beta_t': beta_t_fgls
})
table_4.round(3)

adjust variance of alpha_i using variance of eta_i
0.5163665257131869


,variables,beta_hats,beta_se,beta_t
0,const,3.335,0.154,21.665
1,exp,0.083,0.001,76.022
2,wks,0.001,0.001,2.067
3,bluecol,-0.031,0.014,-2.250
4,ind,0.014,0.015,0.941
5,south,0.006,0.031,0.208
6,smsa,-0.051,0.019,-2.708
7,married,-0.045,0.019,-2.407
8,gender,-0.186,0.097,-1.908
9,union,0.043,0.015,2.908


##Manual - attempt two

In [ ]:
var_alpha_i_adj = (1/N) * sum(alpha_i**2) - (1/T) * var_eta
theta_hat_squared = ( var_eta / (var_eta + T * var_alpha_i_adj) )
X = df.drop(columns=['year', 'i', 'lwage']).to_numpy()
X = np.c_[np.ones(len(X)), X]
y = df['lwage'].to_numpy()
y_avg = P_D @ y
X_avg = P_D @ X

In [ ]:
temp_1 = np.zeros((X.shape[1], X.shape[1]))
temp_2 = np.zeros(X.shape[1])
for i in range(1, 1 + N):
  X_i_star = (1/np.sqrt(var_eta)) * (X[(7*(i-1)):(7*i), :] - (1- np.sqrt(theta_hat_squared)) * X_avg[(7*(i-1)):(7*i), :] )
  y_i_star = (1/np.sqrt(var_eta)) * (y[(7*(i-1)):(7*i)]  - (1- np.sqrt(theta_hat_squared)) * y_avg[(7*(i-1)):(7*i)] )
  temp_1 += X_i_star.T @ X_i_star
  temp_2 += X_i_star.T @ y_i_star
beta_hats_fgls = np.linalg.inv(temp_1) @ temp_2
var_matrix_beta_fgls = np.linalg.inv(temp_1)
beta_se_fgls = np.sqrt(np.diag(var_matrix_beta_fgls))
beta_t_fgls = beta_hats_fgls / beta_se_fgls

In [ ]:
table_4 = pd.DataFrame({
    'variables': ['const'] + df_i.drop(columns = ['year', 'i', 'lwage']).columns.tolist(),
    'beta_hats': beta_hats_fgls,
    'beta_se': beta_se_fgls,
    'beta_t': beta_t_fgls
})
table_4.round(3)

,variables,beta_hats,beta_se,beta_t
0,const,3.335,0.154,21.665
1,exp,0.083,0.001,76.022
2,wks,0.001,0.001,2.067
3,bluecol,-0.031,0.014,-2.250
4,ind,0.014,0.015,0.941
5,south,0.006,0.031,0.208
6,smsa,-0.051,0.019,-2.708
7,married,-0.045,0.019,-2.407
8,gender,-0.186,0.097,-1.908
9,union,0.043,0.015,2.908


##Package
but this one seems to use different parameters, so im not sure if this is comparable

In [ ]:
df_panel = df
df_panel = df_panel.set_index(['i', 'year'])
y_re = df_panel['lwage']
X_re = df_panel.drop(columns=['lwage'])
X_re = sm.add_constant(X_re)
res_re = RandomEffects(y_re, X_re).fit()
print(res_re.summary)

                        RandomEffects Estimation Summary                        
Dep. Variable:                  lwage   R-squared:                        0.3690
Estimator:              RandomEffects   R-squared (Between):             -0.6696
No. Observations:                4165   R-squared (Within):               0.4925
Date:                Wed, Feb 18 2026   R-squared (Overall):             -0.3543
Time:                        09:01:59   Log-likelihood                    752.39
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      220.78
Entities:                         595   P-value                           0.0000
Avg Obs:                       7.0000   Distribution:                 F(11,4153)
Min Obs:                       7.0000                                           
Max Obs:                       7.0000   F-statistic (robust):             220.78
                            

# (d) Hausman Test

In [ ]:
var_matrix_beta_wthn.shape

(8, 8)

In [ ]:
var_matrix_beta_fgls.shape

(12, 12)

shape different, because in the FE(within estimation) we manually dropped all time-invariant regressors.
so we now modify the RE variance matrix to carry out the Hausmann test.

In [ ]:
col_delete = [0]
for varb in time_inv_varb:
  col_index = df.drop(columns = ['lwage', 'year', 'i']).columns.get_loc(varb) + 1 #+1 due to constant
  col_delete.append(col_index)

beta_hats_fgls_modified = np.delete(beta_hats_fgls, col_delete)

var_matrix_beta_fgls_modified = var_matrix_beta_fgls
var_matrix_beta_fgls_modified = np.delete(var_matrix_beta_fgls_modified, col_delete, axis=0)
var_matrix_beta_fgls_modified = np.delete(var_matrix_beta_fgls_modified, col_delete, axis=1)

In [ ]:
print("Is it true that the dimensions of the variance matrix matches after modification?")
var_matrix_beta_wthn.shape == var_matrix_beta_fgls_modified.shape

Is it true that the dimensions of the variance matrix matches after modification?


True

In [ ]:
beta_diff = beta_hats_wthn - beta_hats_fgls_modified
var_diff = var_eta * var_matrix_beta_wthn - var_matrix_beta_fgls_modified

the eigenvalues should be positive, the fgls estimation is likely done incorrectly

In [ ]:
np.linalg.eigvals(var_diff)

array([-9.16022620e-04, -1.16476580e-06, -3.52732719e-07, -3.64126816e-04,
       -3.24001662e-04, -1.67659405e-04, -2.14120925e-04, -2.35002119e-04])

In [ ]:
H = beta_diff.T @ np.linalg.inv(var_diff) @ beta_diff

In [ ]:
H

np.float64(-149.55832075532484)